# IRIS: three-input training in Google Colab
Run cells in order with a **CPU / Python 3.12** runtime. No GPU or GitHub token is needed.

Download the private repository using **Code > Download ZIP**, then upload that ZIP when requested below. The original `IRIS_Model_Code_v2.zip` also works. Use only synthetic or explicitly authorized data.

This notebook trains the ML component. It does not deploy the older backend, start live connectors, execute controls or update an official risk register. Synthetic models remain **shadow/test only**.

In [ ]:
from google.colab import files
from pathlib import Path
import json
import shutil
import subprocess
import sys
import tempfile
import zipfile

print('Python:', sys.version)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Select a Python 3.12 runtime under Runtime > Change runtime type, then rerun this cell.')
uploaded = files.upload()
archives = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one IRIS repository or model-package ZIP.')
WORK = Path(tempfile.mkdtemp(prefix='iris_project_', dir='/content')).resolve()
with zipfile.ZipFile(archives[0]) as archive:
    for member in archive.infolist():
        if not (WORK / member.filename).resolve().is_relative_to(WORK):
            raise ValueError('Unsafe path inside ZIP.')
    archive.extractall(WORK)
candidates = [p.parent for p in WORK.rglob('requirements.txt')
              if (p.parent / 'iris_models' / '__init__.py').is_file()]
if len(candidates) != 1:
    raise ValueError('Could not find exactly one iris_models project in the ZIP.')
PROJECT = candidates[0]
SAMPLES = PROJECT / 'samples' / 'three_inputs'
RUN_ROOT = Path(tempfile.mkdtemp(prefix='iris_run_', dir='/content'))
TRAINING = RUN_ROOT / 'training'
PREDICTIONS = RUN_ROOT / 'prediction_preview'
print('Project:', PROJECT)
print('Fresh run directory:', RUN_ROOT)


## Install the exact dependencies and run the tests
The CLI runs in fresh Python subprocesses, so it does not reuse packages already imported into the notebook kernel. Keep the same pinned dependency versions on the eventual backend.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-q'], cwd=PROJECT, check=True)


## Combine the sources and train both model components
The sources are asset inventory, risk register and SIEM alerts. `training_labels.jsonl` provides **offline answers only**. It is not a fourth runtime input.

The supplied sources are synthetic; keep `--synthetic`. For real data, prepare sessions, obtain independent session/control reviews, use real provenance, and omit that flag. Do not present generated labels or synthetic scores as real-world validation.

In [ ]:
def iris(*arguments):
    return subprocess.run([sys.executable, '-m', 'iris_models', *map(str, arguments)], cwd=PROJECT, check=True)

iris('train-three',
     '--assets', SAMPLES / 'asset_inventory.json',
     '--risks', SAMPLES / 'risk_register.json',
     '--alerts', SAMPLES / 'siem_alerts.jsonl',
     '--labels', SAMPLES / 'training_labels.jsonl',
     '--synthetic', '--out', TRAINING)


In [ ]:
metadata = json.loads((TRAINING / 'models' / 'metadata.json').read_text())
print(json.dumps({
    'selected_scenario_algorithm': metadata['model1']['algorithm'],
    'model1_test_metrics': metadata['model1']['metrics'],
    'model1_release_allowed': metadata['model1']['eligible_for_reviewed_advisory_release'],
    'model2_status': metadata['model2']['status'],
    'model2_release_allowed': metadata['model2']['eligible_for_reviewed_advisory_release']
}, indent=2))
print('Synthetic software check only. Failed or synthetic release gates must not be bypassed.')


## Predict using only the three operational inputs
This makes a **five-session preview**, not a full batch or a new accuracy test. Accuracy is assessed on the held-out split in the evaluation report, not by re-predicting the supplied training data. Remove `--limit 5` below for a full batch.

In [ ]:
iris('predict-three',
     '--assets', SAMPLES / 'asset_inventory.json',
     '--risks', SAMPLES / 'risk_register.json',
     '--alerts', SAMPLES / 'siem_alerts.jsonl',
     '--models', TRAINING / 'models',
     '--mode', 'shadow', '--limit', '5', '--out', PREDICTIONS)
report = json.loads((PREDICTIONS / 'prediction_report.json').read_text())
assert report['training_labels_read'] is False
print({key: report[key] for key in ('total_sessions', 'predictions_written', 'limited_preview', 'training_labels_read', 'automatic_execution')})
with (PREDICTIONS / 'predictions.jsonl').open() as preview:
    print(json.dumps(json.loads(next(preview)), indent=2))


## Download the complete run
Colab storage is temporary. Download the outputs now. Keep the **entire models directory**, reports, original code, matching control catalogue and pinned requirements. Load only artifacts you trained or independently trust: `joblib`/pickle can execute code.

The next engineering step is a backend adapter using the same three-source preparation and session contract. This notebook does not wire that adapter or deploy a website.

In [ ]:
export_path = shutil.make_archive(str(RUN_ROOT) + '_results', 'zip', root_dir=RUN_ROOT)
files.download(export_path)
